first best

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.metrics import make_scorer, accuracy_score, precision_score, recall_score, f1_score
from sklearn.feature_selection import SequentialFeatureSelector
from sklearn.naive_bayes import GaussianNB
from sklearn.svm import SVC, NuSVC
from sklearn.neural_network import MLPClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier

# Chargement des données
df = pd.read_csv("parkinson.csv")
X = df.drop(columns=['ID', 'Recording', 'Status'])
y = df['Status']
X = pd.get_dummies(X, columns=['Gender'], drop_first=True)

# Normalisation
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Initialisation des modèles à évaluer
classifiers = {
    "Naïve Bayes": GaussianNB(),
    "c-SVM": SVC(C=1.0, kernel='rbf'),
    "nu-SVM": NuSVC(nu=0.5, kernel='rbf'),
    "MLP": MLPClassifier(max_iter=2000, random_state=42),
    "KNN": KNeighborsClassifier(n_neighbors=5),
    "Random Forest": RandomForestClassifier(n_estimators=100, random_state=42)
}

# KNN comme base du wrapper (First Best)
base_selector = KNeighborsClassifier(n_neighbors=5)
sfs = SequentialFeatureSelector(base_selector, direction='forward', cv=5, n_jobs=-1)
X_selected = sfs.fit_transform(X_scaled, y)
selected_features_mask = sfs.get_support()

# Évaluation de chaque classifieur sur les features sélectionnées
results = {}

cv = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)
scorers = {
    "Accuracy": make_scorer(accuracy_score),
    "Precision": make_scorer(precision_score),
    "Recall": make_scorer(recall_score),
    "F1-score": make_scorer(f1_score)
}

for name, clf in classifiers.items():
    scores = {metric: [] for metric in scorers}
    for train_idx, test_idx in cv.split(X_selected, y):
        X_train, X_test = X_selected[train_idx], X_selected[test_idx]
        y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]
        clf.fit(X_train, y_train)
        y_pred = clf.predict(X_test)
        for metric, scorer in scorers.items():
            scores[metric].append(scorer._score_func(y_test, y_pred))
    results[name] = {metric: np.mean(vals) for metric, vals in scores.items()}

# Affichage des résultats
results_df = pd.DataFrame(results).T
print("\nRésultats moyens (10-fold CV) avec Wrapper-based (KNN, First Best):")
display(results_df)

# Graphique comparatif
results_df.plot(kind='bar', figsize=(12, 6))
plt.title("Performances des classifieurs avec Wrapper-based FS (KNN, First Best)")
plt.ylabel("Score")
plt.xticks(rotation=45)
plt.grid(True)
plt.tight_layout()
plt.show()


Greed StepWise

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.metrics import make_scorer, accuracy_score, precision_score, recall_score, f1_score
from sklearn.feature_selection import SequentialFeatureSelector
from sklearn.naive_bayes import GaussianNB
from sklearn.svm import SVC, NuSVC
from sklearn.neural_network import MLPClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier

# Chargement des données
df = pd.read_csv("parkinson.csv")
X = df.drop(columns=['ID', 'Recording', 'Status'])
y = df['Status']
X = pd.get_dummies(X, columns=['Gender'], drop_first=True)

# Normalisation
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Initialisation des modèles à évaluer
classifiers = {
    "Naïve Bayes": GaussianNB(),
    "c-SVM": SVC(C=1.0, kernel='rbf'),
    "nu-SVM": NuSVC(nu=0.5, kernel='rbf'),
    "MLP": MLPClassifier(max_iter=2000, random_state=42),
    "KNN": KNeighborsClassifier(n_neighbors=5),
    "Random Forest": RandomForestClassifier(n_estimators=100, random_state=42)
}

# KNN comme base du wrapper (First Best)
base_selector = KNeighborsClassifier(n_neighbors=5)
sfs = SequentialFeatureSelector(base_selector, direction='backward', cv=5, n_jobs=-1)
X_selected = sfs.fit_transform(X_scaled, y)
selected_features_mask = sfs.get_support()

# Évaluation de chaque classifieur sur les features sélectionnées
results = {}

cv = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)
scorers = {
    "Accuracy": make_scorer(accuracy_score),
    "Precision": make_scorer(precision_score),
    "Recall": make_scorer(recall_score),
    "F1-score": make_scorer(f1_score)
}

for name, clf in classifiers.items():
    scores = {metric: [] for metric in scorers}
    for train_idx, test_idx in cv.split(X_selected, y):
        X_train, X_test = X_selected[train_idx], X_selected[test_idx]
        y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]
        clf.fit(X_train, y_train)
        y_pred = clf.predict(X_test)
        for metric, scorer in scorers.items():
            scores[metric].append(scorer._score_func(y_test, y_pred))
    results[name] = {metric: np.mean(vals) for metric, vals in scores.items()}

# Affichage des résultats
results_df = pd.DataFrame(results).T
print("\nRésultats moyens (10-fold CV) avec Wrapper-based (KNN, Greed StepWise):")
display(results_df)

# Graphique comparatif
results_df.plot(kind='bar', figsize=(12, 6))
plt.title("Performances des classifieurs avec Wrapper-based FS (KNN, Greed stepWise)")
plt.ylabel("Score")
plt.xticks(rotation=45)
plt.grid(True)
plt.tight_layout()
plt.show()


PSO

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.naive_bayes import GaussianNB
from sklearn.svm import SVC, NuSVC
from sklearn.neural_network import MLPClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier
import pyswarms as ps

# Chargement et préparation des données
df = pd.read_csv("parkinson.csv")
X = df.drop(columns=['ID', 'Recording', 'Status'])
y = df['Status']
X = pd.get_dummies(X, columns=['Gender'], drop_first=True)
X_scaled = StandardScaler().fit_transform(X)
n_features = X_scaled.shape[1]

# Fonction objectif pour PSO avec base KNN
def objective_function(particles):
    scores = []
    for particle in particles:
        selected = np.where(particle > 0.5)[0]
        if len(selected) == 0:
            scores.append(1.0)  # mauvaise solution
            continue
        X_selected = X_scaled[:, selected]
        cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
        model = KNeighborsClassifier(n_neighbors=5)
        acc = []
        for train_idx, test_idx in cv.split(X_selected, y):
            model.fit(X_selected[train_idx], y.iloc[train_idx])
            preds = model.predict(X_selected[test_idx])
            acc.append(1 - accuracy_score(y.iloc[test_idx], preds))  # 1 - acc (car on minimise)
        scores.append(np.mean(acc))
    return np.array(scores)

# Configuration de PSO
options = {'c1': 2, 'c2': 2, 'w': 0.9, 'k': 5, 'p': 2}  # k = voisins, p = distance
optimizer = ps.discrete.BinaryPSO(n_particles=20, dimensions=n_features, options=options)
cost, best_position = optimizer.optimize(objective_function, iters=30, verbose=True)

# Sélection finale des caractéristiques
selected_features = np.where(best_position > 0.5)[0]
X_final = X_scaled[:, selected_features]

# Classifieurs à évaluer
classifiers = {
    "Naïve Bayes": GaussianNB(),
    "c-SVM": SVC(C=1.0, kernel='rbf'),
    "nu-SVM": NuSVC(nu=0.5, kernel='rbf'),
    "MLP": MLPClassifier(max_iter=2000, random_state=42),
    "KNN": KNeighborsClassifier(n_neighbors=5),
    "Random Forest": RandomForestClassifier(n_estimators=100, random_state=42)
}

# Validation croisée et évaluation
cv = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)
results = {}

for name, clf in classifiers.items():
    accs, precs, recs, f1s = [], [], [], []
    for train_idx, test_idx in cv.split(X_final, y):
        clf.fit(X_final[train_idx], y.iloc[train_idx])
        preds = clf.predict(X_final[test_idx])
        accs.append(accuracy_score(y.iloc[test_idx], preds))
        precs.append(precision_score(y.iloc[test_idx], preds))
        recs.append(recall_score(y.iloc[test_idx], preds))
        f1s.append(f1_score(y.iloc[test_idx], preds))
    results[name] = {
        "Accuracy": np.mean(accs),
        "Precision": np.mean(precs),
        "Recall": np.mean(recs),
        "F1-score": np.mean(f1s)
    }

# Affichage
results_df = pd.DataFrame(results).T
print("\nRésultats avec Wrapper-based PSO (base KNN):")
display(results_df)

# Visualisation
results_df.plot(kind='bar', figsize=(12, 6))
plt.title("Performances des classifieurs avec Wrapper-based FS (KNN, PSO)")
plt.ylabel("Score")
plt.xticks(rotation=45)
plt.grid(True)
plt.tight_layout()
plt.show()
